# 🚁 SWASH-BICOP Auto-RL Trainer

This notebook trains a custom Reinforcement Learning policy for your drone configuration using Google Colab's fast CPUs/GPUs.

### Step 1: Upload the Project
Using the file browser on the left (🗂️ icon), upload the `swash-bicop-initial-version.zip` file containing your local project.
*(Note: You can skip this if you clone directly from a given GitHub URL).* 

Then, run the cells below in order.

In [ ]:
!unzip -q swash-bicop-initial-version.zip
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs

### Step 2: Install Python RL Dependencies & Install App Dependencies

In [ ]:
!cd app-source-v12-pro && npm install
!pip install stable-baselines3 gymnasium websocket-client

### Step 3: Run the Gym Bridge and Train the Policy
This runs the WebSocket server in the background, connects to it, and trains `stable-baselines3` PPO for 50,000 steps using your custom drone physics.


In [ ]:
%%bash 
cd app-source-v12-pro
nohup npx tsx server/gymBridge.ts > gymbridge.log 2>&1 &
sleep 5


In [ ]:
import os
import sys
sys.path.append(os.path.abspath('app-source-v12-pro/server'))

from gym_client import DroneSimEnv
from stable_baselines3 import PPO
from google.colab import files

# ==========================================
# 🔧 EDIT THESE PARAMETERS IF NEEDED
TARGET_MASS = 15.0
TARGET_PROP = 20.0
TARGET_VOLT = 44.4
TRAINING_STEPS = 50000
# ==========================================

print(f"Initialising Gymnasium for mass={TARGET_MASS}kg...")
env = DroneSimEnv(
    host="localhost", 
    port=8765, 
    mass=TARGET_MASS, 
    prop_diameter=TARGET_PROP, 
    battery_voltage=TARGET_VOLT
)

model = PPO("MlpPolicy", env, verbose=1, learning_rate=3e-4, n_steps=2048, batch_size=64, ent_coef=0.01)
print("Starting PPO Training... this may take a few minutes.")
model.learn(total_timesteps=TRAINING_STEPS)

model.save('cargo_policy')
env.close()

print("\n✅ Training complete! Downloading cargo_policy.zip...")
files.download('cargo_policy.zip')
